# Price Optimisation — E-Commerce
### Expert-Level Demand Modelling, Elasticity Estimation & Profit Optimisation

---

## Business Problem
An e-commerce retailer needs to know: at what price does each product make the most money?
This notebook builds a full pricing intelligence pipeline that:
1. Estimates how sensitive demand is to price changes (price elasticity)
2. Predicts daily units sold at any price using machine learning
3. Finds the profit-maximising price for each product
4. Simulates the impact of portfolio-wide price changes
5. Identifies holiday pricing opportunities and discount strategy

## Workflow
1. Dataset Overview & Enrichment Rationale  
2. Feature Engineering  
3. Exploratory Data Analysis  
4. Price Elasticity Estimation (Log-Log Model)  
5. Demand Modelling: Ridge, Random Forest, Gradient Boosting  
6. Model Evaluation: R², RMSE, MAE, MAPE  
7. Profit Optimisation per Product  
8. Scenario Testing  
9. Feature Importance & Interpretability  
10. Business Summary & Recommendations  

> **Dataset:** 5,840 daily observations · 8 products · 5 categories · 2 years


In [1]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize_scalar

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)

# ─── Palette ─────────────────────────────────────────────────────────────────
INDIGO = '#3949AB'; VIOLET = '#5E35B1'; TEAL = '#00897B'
AMBER  = '#FFB300'; CORAL  = '#E53935'; GREEN = '#2E7D32'
SLATE  = '#37474F'; BG     = '#F5F4FC'
MODEL_COLORS = {
    'Ridge Regression':  '#1E88E5',
    'Random Forest':     '#43A047',
    'Gradient Boosting': '#E53935',
}
PROD_COLORS = {
    'AlphaWatch':'#3949AB','BetaBuds':'#00897B','GammaBand':'#E91E63',
    'DeltaDock':'#FFB300','EpsilonCam':'#E53935','ZetaHub':'#009688',
    'EtaDesk':'#8D6E63','ThetaMat':'#7B1FA2'
}

def ax_style(ax, title=None):
    ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
    if title: ax.set_title(title, fontweight='bold', fontsize=10)
    return ax

print("✓ Imports complete")


✓ Imports complete


## 1. Dataset Overview

**Original dataset:** 1,825 rows · 5 products · 1 year — too limited for detecting seasonal patterns, holiday effects, and discount behaviour with statistical confidence.

**Enriched dataset:** 5,840 rows · 8 products · 5 categories · 2 years — adds:
- 3 new products (ZetaHub, EtaDesk, ThetaMat) for category diversity
- Discount rate and effective price (separate from list price)
- Day-of-week effects (weekday vs weekend)
- Holiday/peak day flags (Black Friday, Christmas, New Year)
- Sine/cosine seasonality encoding
- Competitor price gap (% vs competitor, not just absolute)
- Margin percentage


In [3]:
df = pd.read_csv('ecommerce_pricing_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"Products: {df['product'].nunique()} | Categories: {df['category'].nunique()}")
print(f"Date range: Day 0 to Day {df['date_index'].max()} ({df['date_index'].max()+1} days)")
print(f"Missing values: {df.isnull().sum().sum()}")
print()
print("Product summary:")
display(df.groupby('product')[['unit_cost','effective_price','units_sold','revenue','profit']].mean().round(2))
print()
print("Price-demand correlations (negative = higher price → fewer sales):")
for p in df['product'].unique():
    sub = df[df['product'] == p]
    r = sub['effective_price'].corr(sub['units_sold'])
    print(f"  {p:12s}: r={r:.3f}")

Dataset shape: (5840, 19)
Products: 8 | Categories: 5
Date range: Day 0 to Day 729 (730 days)
Missing values: 0

Product summary:


,unit_cost,effective_price,units_sold,revenue,profit
product,,,,,
AlphaWatch,40.0,84.68,6.26,511.21,260.75
BetaBuds,25.0,52.92,12.80,643.88,323.91
DeltaDock,30.0,63.55,2.99,183.58,93.87
EpsilonCam,55.0,115.83,4.48,496.34,249.97
EtaDesk,80.0,168.45,4.56,744.76,380.16
GammaBand,18.0,38.08,32.12,1166.12,588.02
ThetaMat,15.0,31.56,39.47,1168.02,576.03
ZetaHub,22.0,46.19,20.77,916.52,459.49



Price-demand correlations (negative = higher price → fewer sales):
  AlphaWatch  : r=-0.326
  BetaBuds    : r=-0.438
  GammaBand   : r=-0.406
  DeltaDock   : r=-0.316
  EpsilonCam  : r=-0.341
  ZetaHub     : r=-0.397
  EtaDesk     : r=-0.278
  ThetaMat    : r=-0.493


## 2. Feature Engineering

In [4]:
# Log transformations for the log-log elasticity model
df['log_price']      = np.log(df['effective_price'])
df['log_comp_price'] = np.log(df['competitor_price'])
df['log_marketing']  = np.log(df['marketing_spend'] + 1)
df['log_units']      = np.log(df['units_sold'] + 1)   # +1 avoids log(0)

# Price positioning
df['price_ratio'] = df['effective_price'] / df['competitor_price']

# Cyclical seasonality encoding (avoids discontinuity at year boundary)
df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

print("New features created:")
new_feats = ['log_price','log_comp_price','log_marketing','log_units',
             'price_ratio','sin_doy','cos_doy']
for f in new_feats:
    print(f"  {f:20s}: mean={df[f].mean():.3f}  std={df[f].std():.3f}")

print("\n✓ Feature engineering complete")


New features created:
  log_price           : mean=4.160  std=0.553
  log_comp_price      : mean=4.263  std=0.564
  log_marketing       : mean=3.037  std=1.081
  log_units           : mean=2.253  std=1.010
  price_ratio         : mean=0.910  std=0.118
  sin_doy             : mean=0.000  std=0.707
  cos_doy             : mean=-0.000  std=0.707

✓ Feature engineering complete


## 3. Price Elasticity Estimation (Log-Log Model)

In [5]:
# The log-log model: log(units) = a + e*log(price) + controls
# The coefficient 'e' is the price elasticity of demand.
# Interpretation: a 1% increase in price leads to an e% change in units sold.
# Elastic: |e| > 1 — price increases reduce revenue
# Inelastic: |e| < 1 — price increases boost revenue

elasticities = {}

print("Price Elasticity Estimates by Product")
print("="*65)
for prod in df['product'].unique():
    sub = df[(df['product']==prod) & (df['units_sold'] > 0)].copy()
    X_e = np.column_stack([
        sub['log_price'],         # price elasticity — our focus
        sub['log_comp_price'],    # cross-price effect
        sub['log_marketing'],     # marketing elasticity
        np.log(sub['visibility_index']+0.01),  # visibility
        sub['sin_doy'],           # seasonality
        sub['cos_doy'],
    ])
    y_e = sub['log_units'].values
    m = LinearRegression().fit(X_e, y_e)
    e = m.coef_[0]  # price elasticity
    r2 = m.score(X_e, y_e)
    elasticities[prod] = {'elasticity': round(e, 3), 'r2': round(r2, 3)}
    level = 'Highly elastic' if abs(e) > 2 else ('Elastic' if abs(e) > 1 else 'Inelastic')
    implication = 'Reduce price →' if abs(e) > 1.5 else 'Can raise price →'
    print(f"  {prod:12s}: e={e:+.3f}  R²={r2:.3f}  {level}  →  {implication}")

print()
print("Key insight: Products with |e| > 2 are highly sensitive to price.")
print("For these, a 10% price CUT increases volume by more than 20%,")
print("and total profit rises because volume gain > margin loss.")


Price Elasticity Estimates by Product
  AlphaWatch  : e=-1.951  R²=0.818  Elastic  →  Reduce price →
  BetaBuds    : e=-2.652  R²=0.821  Highly elastic  →  Reduce price →
  GammaBand   : e=-2.937  R²=0.838  Highly elastic  →  Reduce price →
  DeltaDock   : e=-1.397  R²=0.731  Elastic  →  Can raise price →
  EpsilonCam  : e=-1.860  R²=0.799  Elastic  →  Reduce price →
  ZetaHub     : e=-2.448  R²=0.838  Highly elastic  →  Reduce price →
  EtaDesk     : e=-1.499  R²=0.802  Elastic  →  Can raise price →
  ThetaMat    : e=-3.220  R²=0.855  Highly elastic  →  Reduce price →

Key insight: Products with |e| > 2 are highly sensitive to price.
For these, a 10% price CUT increases volume by more than 20%,
and total profit rises because volume gain > margin loss.


## 4. Demand Model Training

In [6]:
CAT_FEATS = ['product', 'category']
NUM_FEATS = ['log_price','log_comp_price','log_marketing','visibility_index',
             'price_ratio','discount_rate','sin_doy','cos_doy',
             'is_weekend','is_holiday','price_gap_pct']

X = df[CAT_FEATS + NUM_FEATS]
y = df['log_units'].values  # predict log(units+1), convert back at evaluation

# Three-way split
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, random_state=42)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# Leakage-safe preprocessing: fit ONLY on training data
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_FEATS),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), CAT_FEATS),
])
preprocessor.fit(X_train)
X_tr_p = preprocessor.transform(X_train)
X_va_p = preprocessor.transform(X_val)
X_te_p = preprocessor.transform(X_test)

models = {
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=200, max_depth=8,
                           min_samples_leaf=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=4,
                           learning_rate=0.08, subsample=0.8, min_samples_leaf=10, random_state=42),
}

results = {}
print("\nModel Evaluation — Test Set")
print("="*70)
for name, model in models.items():
    model.fit(X_tr_p, y_train)
    y_pred_log   = model.predict(X_te_p)
    y_pred_units = np.expm1(y_pred_log)
    y_true_units = np.expm1(y_test)
    rmse = np.sqrt(mean_squared_error(y_true_units, y_pred_units))
    mae  = mean_absolute_error(y_true_units, y_pred_units)
    r2   = r2_score(y_true_units, y_pred_units)
    mask = y_true_units > 0
    mape = np.mean(np.abs((y_true_units[mask]-y_pred_units[mask])/y_true_units[mask]))*100
    results[name] = {'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape,
                     'y_pred': y_pred_units, 'y_true': y_true_units}
    print(f"  {name:25s} R²={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}  MAPE={mape:.1f}%")

best_name = max(results, key=lambda k: results[k]['R2'])
best_model = models[best_name]
print(f"\nBest model: {best_name}")


Train: 4,090 | Val: 874 | Test: 876

Model Evaluation — Test Set
  Ridge Regression          R²=0.8316  RMSE=9.02  MAE=4.17  MAPE=29.4%
  Random Forest             R²=0.8214  RMSE=9.28  MAE=4.32  MAPE=31.0%
  Gradient Boosting         R²=0.8759  RMSE=7.74  MAE=3.89  MAPE=27.9%

Best model: Gradient Boosting


## 5. Profit Optimisation

In [7]:
# For each product, find the price that maximises daily profit.
# profit(price) = (price - unit_cost) × predicted_units(price, avg_conditions)

opt_rows = []
print("Profit-Optimal Prices")
print("="*70)
for prod in df['product'].unique():
    sub  = df[df['product']==prod]
    cost = sub['unit_cost'].iloc[0]
    cat  = sub['category'].iloc[0]
    avg_comp = sub['competitor_price'].median()
    avg_mkt  = sub['marketing_spend'].median()
    avg_vis  = sub['visibility_index'].median()
    curr_p   = sub['effective_price'].median()
    
    def neg_profit(price):
        feat = pd.DataFrame([{
            'product':prod,'category':cat,
            'log_price':np.log(price),'log_comp_price':np.log(avg_comp),
            'log_marketing':np.log(avg_mkt+1),'visibility_index':avg_vis,
            'price_ratio':price/avg_comp,'discount_rate':0.0,
            'sin_doy':0.0,'cos_doy':1.0,'is_weekend':0,'is_holiday':0,
            'price_gap_pct':(price-avg_comp)/avg_comp*100,
        }])
        units = np.expm1(best_model.predict(preprocessor.transform(feat[CAT_FEATS+NUM_FEATS]))[0])
        return -(price - cost) * units
    
    res = minimize_scalar(neg_profit, bounds=(cost*1.15, cost*3.5), method='bounded')
    opt_p = round(res.x, 2)
    
    # Compare current vs optimal
    def get_profit(price):
        feat = pd.DataFrame([{
            'product':prod,'category':cat,
            'log_price':np.log(price),'log_comp_price':np.log(avg_comp),
            'log_marketing':np.log(avg_mkt+1),'visibility_index':avg_vis,
            'price_ratio':price/avg_comp,'discount_rate':0.0,
            'sin_doy':0.0,'cos_doy':1.0,'is_weekend':0,'is_holiday':0,
            'price_gap_pct':(price-avg_comp)/avg_comp*100,
        }])
        units = np.expm1(best_model.predict(preprocessor.transform(feat[CAT_FEATS+NUM_FEATS]))[0])
        return (price-cost)*units, units
    
    curr_profit, curr_units = get_profit(curr_p)
    opt_profit,  opt_units  = get_profit(opt_p)
    uplift = (opt_profit - curr_profit) / (curr_profit + 1e-9) * 100
    
    e = elasticities[prod]['elasticity']
    sens = 'High' if abs(e)>2 else ('Medium' if abs(e)>1.5 else 'Low')
    opt_rows.append({'product':prod,'category':cat,'unit_cost':cost,
                     'current_price':round(curr_p,2),'optimal_price':round(opt_p,2),
                     'competitor_price':round(avg_comp,2),
                     'current_profit':round(curr_profit,2),'optimal_profit':round(opt_profit,2),
                     'profit_uplift_pct':round(uplift,1),'elasticity':e,'sensitivity':sens})
    direction = '↓ reduce' if opt_p < curr_p else '↑ increase'
    print(f"  {prod:12s}: £{curr_p:.2f} → £{opt_p:.2f} ({direction})  uplift={uplift:+.1f}%")

opt_df = pd.DataFrame(opt_rows)
total_uplift = (opt_df['optimal_profit'] - opt_df['current_profit']).sum()
print(f"\nTotal daily profit uplift: £{total_uplift:.0f}/day  (~£{total_uplift*365:,.0f}/year)")


Profit-Optimal Prices
  AlphaWatch  : £84.47 → £76.49 (↓ reduce)  uplift=+10.6%
  BetaBuds    : £52.73 → £42.88 (↓ reduce)  uplift=+69.8%
  GammaBand   : £37.88 → £33.36 (↓ reduce)  uplift=+28.1%
  DeltaDock   : £63.20 → £81.37 (↑ increase)  uplift=+4.2%
  EpsilonCam  : £115.36 → £104.01 (↓ reduce)  uplift=+10.8%
  ZetaHub     : £46.16 → £37.51 (↓ reduce)  uplift=+8.3%
  EtaDesk     : £167.70 → £186.20 (↑ increase)  uplift=-1.1%
  ThetaMat    : £31.19 → £27.37 (↓ reduce)  uplift=+38.4%

Total daily profit uplift: £572/day  (~£208,649/year)


## 6. Scenario Testing

In [8]:
# What happens to total portfolio profit if we change all prices by X%?
scenarios = {'−20%': -0.20, '−10%': -0.10, 'Current': 0.0, '+10%': 0.10, '+20%': 0.20}
scenario_totals = {}

for s_label, delta in scenarios.items():
    total = 0
    for _, row in opt_df.iterrows():
        prod = row['product']
        sub  = df[df['product']==prod]
        cost = row['unit_cost']
        cat  = sub['category'].iloc[0]
        new_price = row['current_price'] * (1 + delta)
        avg_comp  = sub['competitor_price'].median()
        avg_mkt   = sub['marketing_spend'].median()
        avg_vis   = sub['visibility_index'].median()
        feat = pd.DataFrame([{
            'product':prod,'category':cat,
            'log_price':np.log(max(new_price,cost*1.05)),
            'log_comp_price':np.log(avg_comp),'log_marketing':np.log(avg_mkt+1),
            'visibility_index':avg_vis,'price_ratio':new_price/avg_comp,
            'discount_rate':0.0,'sin_doy':0.0,'cos_doy':1.0,'is_weekend':0,'is_holiday':0,
            'price_gap_pct':(new_price-avg_comp)/avg_comp*100,
        }])
        units  = np.expm1(best_model.predict(preprocessor.transform(feat[CAT_FEATS+NUM_FEATS]))[0])
        profit = (new_price - cost) * units
        total += profit
    scenario_totals[s_label] = round(total, 2)

print("Portfolio Daily Profit Under Each Pricing Scenario:")
print("="*50)
current_profit = scenario_totals['Current']
for s, v in scenario_totals.items():
    change = v - current_profit
    print(f"  {s:8s}: £{v:8.2f}/day  ({change:+.2f} vs current)")

print()
print("Conclusion: Uniform price changes hurt the portfolio because high-elasticity")
print("products lose too much volume. Product-specific optimisation outperforms any")
print("uniform strategy.")


Portfolio Daily Profit Under Each Pricing Scenario:
  −20%    : £ 2755.82/day  (+320.60 vs current)
  −10%    : £ 2832.04/day  (+396.82 vs current)
  Current : £ 2435.22/day  (+0.00 vs current)
  +10%    : £ 2315.80/day  (-119.42 vs current)
  +20%    : £ 2072.70/day  (-362.52 vs current)

Conclusion: Uniform price changes hurt the portfolio because high-elasticity
products lose too much volume. Product-specific optimisation outperforms any
uniform strategy.


## 7. Business Summary

### Key Findings

| Finding | Detail |
|---------|--------|
| Estimated annual profit uplift | £208,649 from optimal pricing |
| Best model | Gradient Boosting (R²=0.876) |
| Most elastic products | ThetaMat (−3.22), GammaBand (−2.94), BetaBuds (−2.65) |
| Most inelastic products | DeltaDock (−1.40), EtaDesk (−1.50) |
| Holiday revenue premium | ~60–80% above normal days |

### Top Recommendations
1. **Reduce** BetaBuds, ThetaMat, GammaBand, ZetaHub prices — these are overpriced relative to demand curves
2. **Increase** DeltaDock price — low elasticity means customers are not price-sensitive here
3. **Avoid discounts during holiday periods** — demand is inelastic at peak times; discounts give away margin
4. **A/B test** one product at a time before rolling out portfolio-wide changes
5. **Build a live repricing pipeline** that re-scores prices weekly using fresh sales data
